In [1]:
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../dataset_metadata.csv")
print(len(df))

1440


In [3]:
def add_noise(audio):
    noise = np.random.normal(0, 0.02, audio.shape)
    return audio + noise

def pitch_shift(audio, sr):
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=2)

def time_stretch(audio):
    return librosa.effects.time_stretch(audio, rate=0.9)

def extract_logmel_from_audio(audio, sr):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128,
        hop_length=512
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel

In [4]:
X=[]
y=[]
max_len = 128
for index, row in tqdm(df.iterrows(), total=len(df)):
    audio, sr = librosa.load(row["path"], sr=22050)
    #Original + Augmented versions
    variants = [audio, add_noise(audio), pitch_shift(audio, sr)]
    for variant in variants:
        spec = extract_logmel_from_audio(variant, sr)
        #pad or truncate for inputs to have identical shape
        if spec.shape[1] < max_len:
            pad_width = max_len - spec.shape[1]
            spec = np.pad(spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            spec = spec[:, :max_len]
        X.append(spec)
        y.append(row["emotion"])

X = np.array(X)
y = np.array(y)

print("Shape:",X.shape)

  0%|                                                  | 0/1440 [00:00<?, ?it/s]C:\ddata\git program\speech-emotion-recognition\ser_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|███████████████████████████████████████| 1440/1440 [06:41<00:00,  3.58it/s]


Shape: (4320, 128, 128)


In [6]:
X = X[..., np.newaxis]
print(X.shape)

(4320, 128, 128, 1, 1)


In [7]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [9]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / (std + 1e-6)
X_test = (X_test - mean) / (std + 1e-6)

In [11]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Reshape, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

In [13]:
input_layer = Input(shape=(128,128,1))

x = Conv2D(16, (3,3), activation='relu')(input_layer)
x = MaxPooling2D((2,2))(x)

x = Conv2D(32, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)

# shape now: (None, 30, 30, 32)

from tensorflow.keras.layers import Permute

x = Permute((2,1,3))(x)
x = Reshape((30, 30*32))(x)

x = LSTM(64, return_sequences=False)(x)

x = Dense(64, activation='relu',
          kernel_regularizer=regularizers.l2(0.001))(x)

x = Dropout(0.5)(x)

output_layer = Dense(8, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape            ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 1)     │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 16)    │           160 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 16)      │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 32)      │         4,640 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 32)      │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 30, 30, 32)      │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 30, 960)         │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)              │       262,400 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)              │         4,160 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)              │             0 │
├─────────────────────────────────┼─────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)               │           520 │
└─────────────────────────────────┴─────────────────────────┴───────────────┘

 Total params: 271,880 (1.04 MB)

 Trainable params: 271,880 (1.04 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [15]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 209ms/step - accuracy: 0.1820 - loss: 2.0881 - val_accuracy: 0.2688 - val_loss: 1.9832
Epoch 2/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 15s 166ms/step - accuracy: 0.2406 - loss: 1.9802 - val_accuracy: 0.2890 - val_loss: 1.8710
Epoch 3/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 25s 212ms/step - accuracy: 0.2757 - loss: 1.8949 - val_accuracy: 0.3107 - val_loss: 1.8373
Epoch 4/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 15s 175ms/step - accuracy: 0.2873 - loss: 1.8480 - val_accuracy: 0.3468 - val_loss: 1.7310
Epoch 5/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 13s 154ms/step - accuracy: 0.3184 - loss: 1.7857 - val_accuracy: 0.3483 - val_loss: 1.7240
Epoch 6/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 28s 237ms/step - accuracy: 0.3517 - loss: 1.7396 - val_accuracy: 0.3916 - val_loss: 1.6605
Epoch 7/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 30s 345ms/step - accuracy: 0.3723 - loss: 1.6707 - val_accuracy: 0.3671 - val_loss: 1.6902
Epoch 8/40
87/87 ━━━━━━━━━━━━━━━━━━━━ 28s 318ms/step - accuracy: 0.4085 - loss: 1.6034 - val_accu

In [16]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Hybrid Test Accuracy:", test_acc)

27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.6377 - loss: 1.0588
Hybrid Test Accuracy: 0.6377314925193787
